In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
DGA Domain Detection - CPU Version (Fβ=0.5 ≈ 0.91)
Based on working dga-study.ipynb
"""

import numpy as np
import gc
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, fbeta_score
from math import log2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
RANDOM_STATE = 42
TEST_SIZE = 0.2
CATBOOST_ITERATIONS = 1500
CATBOOST_LR = 0.1
CATBOOST_DEPTH = 6
EARLY_STOPPING_ROUNDS = 150
BETA = 0.5

# English letter frequencies
ENGLISH_FREQ = {
    'e': 0.127, 't': 0.091, 'a': 0.082, 'o': 0.075, 'i': 0.070,
    'n': 0.069, 's': 0.063, 'h': 0.061, 'r': 0.060, 'd': 0.043,
    'l': 0.040, 'c': 0.028, 'u': 0.028, 'm': 0.024, 'w': 0.024,
    'f': 0.022, 'g': 0.020, 'y': 0.020, 'p': 0.019, 'b': 0.015,
    'v': 0.010, 'k': 0.008, 'j': 0.002, 'x': 0.002, 'q': 0.001, 'z': 0.001
}

# ============================================================================
# FEATURE FUNCTIONS
# ============================================================================

def calculate_entropy(s):
    """Shannon entropy - measures randomness."""
    if len(s) == 0:
        return 0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * log2(p) for p in probs)


def chi_square_score(name):
    """Chi-Square test vs English letter distribution."""
    if len(name) == 0:
        return 0
    observed = {}
    for c in name:
        observed[c] = observed.get(c, 0) + 1
    
    chi_sq = 0
    for c in observed:
        expected = ENGLISH_FREQ.get(c, 0.001) * len(name)
        chi_sq += ((observed[c] - expected) ** 2) / max(expected, 0.001)
    
    return chi_sq / len(name)


def count_vowel_clusters(name):
    """Count max consecutive vowels."""
    vowels = set('aeiouy')
    max_vowel_cluster = 0
    current_cluster = 0
    
    for c in name:
        if c in vowels:
            current_cluster += 1
            max_vowel_cluster = max(max_vowel_cluster, current_cluster)
        else:
            current_cluster = 0
    
    return max_vowel_cluster


def extract_advanced_features(domain, popular_domains=None):
    """Extract 28 features from domain name."""
    if not isinstance(domain, str) or not domain:
        return np.zeros(28, dtype=np.float32)

    # Clean domain
    clean_domain = domain.lower().strip()
    clean_domain = clean_domain.replace('http://', '').replace('https://', '')
    clean_domain = clean_domain.replace('www.', '')
    clean_domain = clean_domain.split('/')[0]
    
    # Split name and TLD
    parts = clean_domain.rsplit('.', 1)
    name = parts[0].lower()
    tld = parts[1].lower() if len(parts) > 1 else ""

    # Popularity check
    is_popular = 0
    if popular_domains is not None:
        if clean_domain in popular_domains or name in popular_domains:
            is_popular = 1
    
    # Basic features
    length = len(name)
    tld_length = len(tld)
    
    # Character composition
    digits = sum(c.isdigit() for c in name)
    letters = sum(c.isalpha() for c in name)
    special_chars = length - digits - letters
    
    digit_ratio = digits / max(1, length)
    letter_ratio = letters / max(1, length)
    
    # Vowels/Consonants
    vowels = set('aeiouy')
    consonants = set('bcdfghjklmnpqrstvwxz')
    
    v_count = sum(c in vowels for c in name)
    c_count = sum(c in consonants for c in name)
    
    vowel_ratio = v_count / max(1, length)
    consonant_ratio = c_count / max(1, length)
    vc_ratio = v_count / max(1, c_count)
    
    # Entropy
    entropy = calculate_entropy(name)

    # Rare bigrams
    if length > 1:
        bigrams = [name[i:i+2] for i in range(len(name)-1)]
        bigram_counts = {}
        for bg in bigrams:
            bigram_counts[bg] = bigram_counts.get(bg, 0) + 1
        rare_bigrams = sum(1 for count in bigram_counts.values() if count == 1)
        rare_bigram_ratio = rare_bigrams / max(1, len(bigrams))
    else:
        rare_bigram_ratio = 0

    # Rare trigrams
    if length > 2:
        trigrams = [name[i:i+3] for i in range(len(name)-2)]
        trigram_counts = {}
        for tg in trigrams:
            trigram_counts[tg] = trigram_counts.get(tg, 0) + 1
        rare_trigrams = sum(1 for count in trigram_counts.values() if count == 1)
        rare_trigram_ratio = rare_trigrams / max(1, len(trigrams))
    else:
        rare_trigram_ratio = 0

    # Consonant streaks
    has_4_consonants = 0
    has_5_consonants = 0
    consonant_streak = 0
    max_consonant_streak = 0
    
    for c in name:
        if c in consonants:
            consonant_streak += 1
            max_consonant_streak = max(max_consonant_streak, consonant_streak)
            if consonant_streak >= 4:
                has_4_consonants = 1
            if consonant_streak >= 5:
                has_5_consonants = 1
        else:
            consonant_streak = 0
    
    # Vowel clusters
    max_vowel_cluster = count_vowel_clusters(name)
    has_3_vowels = int(max_vowel_cluster >= 3)
    
    # Chi-Square
    chi_sq = chi_square_score(name)
    
    # Bigram ratio
    if length > 1:
        bigrams = [name[i:i+2] for i in range(len(name)-1)]
        unique_bigrams = len(set(bigrams))
        bigram_ratio = unique_bigrams / max(1, len(bigrams))
    else:
        bigram_ratio = 0
        
    # Max consecutive identical chars
    max_consecutive = 1
    current_consecutive = 1
    for i in range(1, len(name)):
        if name[i] == name[i-1]:
            current_consecutive += 1
            max_consecutive = max(max_consecutive, current_consecutive)
        else:
            current_consecutive = 1
            
    # Digit position features
    has_digits = int(digits > 0)
    starts_with_digit = int(name[0].isdigit()) if length > 0 else 0
    ends_with_digit = int(name[-1].isdigit()) if length > 0 else 0
    
    # TLD features
    popular_tlds = {'com', 'org', 'net', 'ru', 'de', 'uk', 'info', 'biz'}
    is_popular_tld = int(tld in popular_tlds)
    
    # Feature vector (28 features)
    features = [
        length,                      # 0
        tld_length,                  # 1
        digits,                      # 2
        digit_ratio,                 # 3
        letter_ratio,                # 4
        v_count,                     # 5
        c_count,                     # 6
        vowel_ratio,                 # 7
        consonant_ratio,             # 8
        vc_ratio,                    # 9
        entropy,                     # 10
        rare_bigram_ratio,           # 11
        rare_trigram_ratio,          # 12
        has_4_consonants,            # 13
        has_5_consonants,            # 14
        max_vowel_cluster,           # 15
        has_3_vowels,                # 16
        chi_sq,                      # 17
        max_consonant_streak,        # 18
        max_consecutive,             # 19
        has_digits,                  # 20
        starts_with_digit,           # 21
        ends_with_digit,             # 22
        is_popular_tld,              # 23
        special_chars,               # 24
        len(set(name)),              # 25
        length / max(1, len(set(name))), # 26
        is_popular                   # 27
    ]
    
    return np.array(features, dtype=np.float32)


def load_popular_domains(filepath):
    """Load popular domains dictionary."""
    popular_set = set()
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                domain = line.strip().lower()
                if domain:
                    domain = domain.replace('http://', '').replace('https://', '')
                    domain = domain.replace('www.', '').split('/')[0]
                    popular_set.add(domain)
        print(f"✓ Loaded {len(popular_set):,} popular domains")
        return popular_set
    except FileNotFoundError:
        print(f"⚠ Dictionary not found: {filepath}")
        return set()


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("=" * 70)
    print("DGA DOMAIN DETECTION - CPU SOLUTION")
    print("=" * 70)
    
    # --- Load popular domains ---
    print("\n[1/7] Loading popular domains...")
    popular_domains = load_popular_domains('/kaggle/input/datasets/shvachko/dictionary/web.txt')
    
    # --- Load training data ---
    print("\n[2/7] Loading training data...")
    train = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/train.csv')
    train['domain'] = train['domain'].fillna('').astype(str)
    print(f"✓ Train: {len(train):,} samples | Labels: {train['label'].value_counts().to_dict()}")
    
    # --- Feature extraction (train) ---
    print("\n[3/7] Extracting train features...")
    X_train = np.array([
        extract_advanced_features(d, popular_domains) 
        for d in tqdm(train['domain'], desc="Train")
    ], dtype=np.float32)
    y_train = train['label'].values
    del train
    gc.collect()
    print(f"✓ Train features: {X_train.shape}")
    
    # --- Load test data ---
    print("\n[4/7] Loading test data...")
    test = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/test.csv')
    test_ids = test['id'].values.copy() if 'id' in test.columns else np.arange(len(test))
    test['domain'] = test['domain'].fillna('').astype(str)
    print(f"✓ Test: {len(test):,} samples")
    
    # --- Feature extraction (test) ---
    print("\n[5/7] Extracting test features...")
    X_test = np.array([
        extract_advanced_features(d, popular_domains) 
        for d in tqdm(test['domain'], desc="Test")
    ], dtype=np.float32)
    del test
    gc.collect()
    print(f"✓ Test features: {X_test.shape}")
    
    # --- Train/Validation split ---
    print("\n[6/7] Training CatBoost model (CPU)...")
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, 
        test_size=TEST_SIZE, 
        random_state=RANDOM_STATE, 
        stratify=y_train
    )
    
    # Class balance
    n_neg = np.sum(y_tr == 0)
    n_pos = np.sum(y_tr == 1)
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
    print(f"  → Classes: 0={n_neg:,}, 1={n_pos:,} | weight={scale_pos_weight:.3f}")
    
    # CatBoost params - CPU MODE
    params = {
        'iterations': CATBOOST_ITERATIONS,
        'learning_rate': CATBOOST_LR,
        'depth': CATBOOST_DEPTH,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'task_type': 'CPU',      # ← КЛЮЧЕВОЕ ИЗМЕНЕНИЕ
        'verbose': 50,
        'random_seed': RANDOM_STATE,
        'scale_pos_weight': scale_pos_weight,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
        'use_best_model': True,
        'thread_count': -1       # Use all CPU cores
    }
    
    print("  → Starting training (this may take 30-60 min on CPU)...")
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), plot=False)
    
    # --- Validation evaluation ---
    print("\n[7/7] Evaluating model...")
    y_val_pred = model.predict(X_val)
    
    accuracy = accuracy_score(y_val, y_val_pred)
    f_beta = fbeta_score(y_val, y_val_pred, beta=BETA)
    
    print(f"\n✓ Validation Results:")
    print(f"  → Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  → F-{BETA} Score: {f_beta:.4f} ← TARGET")
    print(f"\n  Classification Report:")
    print(classification_report(y_val, y_val_pred, digits=4))
    print(f"\n  Confusion Matrix:\n{confusion_matrix(y_val, y_val_pred)}")
    
    # --- Feature importance ---
    feature_names = [
        'length', 'tld_length', 'digits', 'digit_ratio', 'letter_ratio',
        'v_count', 'c_count', 'vowel_ratio', 'consonant_ratio', 'vc_ratio',
        'entropy', 'rare_bigram_ratio', 'rare_trigram_ratio',
        'has_4_consonants', 'has_5_consonants', 'max_vowel_cluster', 'has_3_vowels',
        'chi_square', 'max_consonant_streak', 'max_consecutive',
        'has_digits', 'starts_with_digit', 'ends_with_digit', 'is_popular_tld',
        'special_chars', 'unique_chars', 'char_density', 'is_popular'
    ]
    
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    print(f"\n✓ Top-10 Features:")
    print("-" * 50)
    for i in sorted_idx[:10]:
        print(f"  {feature_names[i]:<25} {importances[i]:>8.3f}")
    
    # --- Generate submission ---
    print("\n" + "=" * 70)
    print("GENERATING SUBMISSION")
    print("=" * 70)
    
    y_test_pred = model.predict(X_test)
    
    submission = pd.DataFrame({'id': test_ids, 'label': y_test_pred.astype(int)})
    submission.to_csv('submission.csv', index=False)
    
    pred_dist = {0: np.sum(y_test_pred == 0), 1: np.sum(y_test_pred == 1)}
    print(f"\n✓ Saved: submission.csv")
    print(f"  → Predictions: Class 0={pred_dist[0]:,}, Class 1={pred_dist[1]:,}")
    print(f"\n🎉 DONE! 🎉")
    
    return f_beta


if __name__ == "__main__":
    result = main()
    print(f"\nFinal F-{BETA} score: {result:.4f}")

DGA DOMAIN DETECTION - CPU SOLUTION

[1/7] Loading popular domains...
⚠ Dictionary not found: /kaggle/input/datasets/shvachko/dictionary/web.txt

[2/7] Loading training data...
✓ Train: 17,719,790 samples | Labels: {0: 9838485, 1: 7881305}

[3/7] Extracting train features...


Train: 100%|██████████| 17719790/17719790 [15:25<00:00, 19155.35it/s]


✓ Train features: (17719790, 28)

[4/7] Loading test data...
✓ Test: 7,594,197 samples

[5/7] Extracting test features...


Test: 100%|██████████| 7594197/7594197 [07:30<00:00, 16851.26it/s]


✓ Test features: (7594197, 28)

[6/7] Training CatBoost model (CPU)...
  → Classes: 0=7,870,788, 1=6,305,044 | weight=1.248
  → Starting training (this may take 30-60 min on CPU)...
0:	test: 0.9096722	best: 0.9096722 (0)	total: 3.02s	remaining: 1h 15m 23s
50:	test: 0.9611540	best: 0.9611540 (50)	total: 2m 14s	remaining: 1h 3m 31s
100:	test: 0.9669441	best: 0.9669441 (100)	total: 4m 24s	remaining: 1h 1m 9s
150:	test: 0.9692686	best: 0.9692686 (150)	total: 6m 33s	remaining: 58m 35s
200:	test: 0.9705897	best: 0.9705897 (200)	total: 8m 42s	remaining: 56m 17s
250:	test: 0.9716592	best: 0.9716592 (250)	total: 10m 50s	remaining: 53m 55s
300:	test: 0.9724329	best: 0.9724329 (300)	total: 12m 56s	remaining: 51m 34s
350:	test: 0.9730896	best: 0.9730896 (350)	total: 15m 4s	remaining: 49m 21s
400:	test: 0.9736170	best: 0.9736170 (400)	total: 17m 11s	remaining: 47m 7s
450:	test: 0.9740556	best: 0.9740556 (450)	total: 19m 19s	remaining: 44m 57s
500:	test: 0.9744610	best: 0.9744610 (500)	total: 21m 28